# KNX — group addresses, tunnelling and an office floor

A KNX IP router with an office floor on its line shares an in-memory UDP network with a tunnelling client. Point the client at a real interface with `UseGateway("192.168.1.10")`; it stays read-only until `AllowWrites()`.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.27.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.27.0-preview.1"

## Datapoint types and telegrams
A group address carries one datapoint type; the KNX 2-byte float holds temperatures, and small values ride in the APCI byte.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Knx;
using IoTCom.Net.Transports;
using System.Net;

foreach (var (dpt, value) in new (string, object)[] { ("1.001", true), ("5.001", 70.0), ("9.001", 21.5), ("9.001", -30.0), ("14.056", 1234.5) })
    Console.WriteLine($"{dpt,-7} {value,-7} -> {KnxDpt.Encode(dpt, value),-14} {KnxDpt.Format(dpt, KnxDpt.Encode(dpt, value))}");
var telegram = Convert.FromHexString("2900BCE011041801030080 0C33".Replace(" ", ""));
var frame = KnxCemiFrame.Decode(telegram);
Console.WriteLine($"{frame} = {KnxDpt.Format("9.001", frame.Value)}");
foreach (var f in KnxCemiFrame.Describe(telegram)) Console.WriteLine($"  {f.Name,-13} {f.Value}");

## Find the router and open a tunnel
SEARCH_REQUEST goes to the multicast group; CONNECT_REQUEST returns a channel and an individual address.

In [ ]:
var knxNet = new InMemoryDatagramNetwork();
var routerAddress = new IPEndPoint(IPAddress.Parse("10.50.0.2"), KnxIp.Port);
var knxBus = new KnxBus();
var floor = new KnxBuildingSimulator(knxBus);
var router = KnxIpGateway.Create(o => { o.UseInMemory(knxNet, routerAddress); o.Bus = knxBus; });
await router.StartAsync();
foreach (var d in await KnxDiscovery.SearchAsync(knxNet.Bind(), TimeSpan.FromMilliseconds(300))) Console.WriteLine(d);
var tunnel = KnxTunnelClient.Create(o => o.UseInMemory(knxNet).UseGateway(routerAddress));
await tunnel.ConnectAsync();
Console.WriteLine($"tunnel channel {tunnel.Channel}, address {tunnel.Address}");
foreach (var ga in new[] { "3/0/1", "3/1/1", "1/1/1", "4/1/0" })
{
    var info = floor.Find(KnxGroupAddress.Parse(ga))!;
    Console.WriteLine($"{ga,-6} {info.Name,-28} {KnxDpt.Format(info.Dpt, await tunnel.ReadAsync(KnxGroupAddress.Parse(ga)))}");
}

## Watch the line, then write after opting in
The tunnel reports every telegram; this one is read-only, so a second, write-enabled tunnel switches a light and moves a blind.

In [ ]:
var seen = new List<string>();
tunnel.GroupValueReceived += e => { var i = floor.Find(e.Destination); lock (seen) seen.Add($"{e.Source} -> {e.Destination} {(i is null ? e.Value.ToString() : KnxDpt.Format(i.Dpt, e.Value))} {i?.Name}"); };
try { await tunnel.WriteAsync("1/0/1", "1.001", true); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
var writer = KnxTunnelClient.Create(o => o.UseInMemory(knxNet).UseGateway(routerAddress).AllowWrites());
await writer.ConnectAsync();
await writer.WriteAsync("1/0/1", "1.001", true);
await writer.WriteAsync("2/1/1", "5.001", 60);
for (var i = 0; i < 4; i++) { floor.Step(5); await Task.Delay(100); }
lock (seen) foreach (var line in seen.Take(12)) Console.WriteLine(line);
Console.WriteLine($"{floor.Rooms[0].Name}: light {floor.Rooms[0].LightOn}, blind {floor.Rooms[0].Blind:0} %");
await writer.DisposeAsync();
await tunnel.DisposeAsync();
await router.DisposeAsync();
await knxBus.DisposeAsync();

## Going further
`iotcom knx demo`, `iotcom knx discover`, `iotcom knx monitor --gateway <host>`, `iotcom knx read 3/0/1 --sim`, `iotcom knx decode <hex>`, the Gallery's *Office floor over KNX* and the KnxMonitor sample. See `docs/en/protocols/knx.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*